# Phase 2 — precompute: OCR + frozen SigLIP embeddings

Launcher for Phase 2 of Laya-Vision (see `docs/PROJECT_SPEC.md`, `DECISIONS.md`). A one-off pass over all 70,000
images: Tesseract OCR on the original scan, preprocessing p2v1 (auto-contrast → long side 384 → centre-pad white),
SigLIP-base-384 patch tokens pooled 8×8 = 64 tokens, stored fp16 in ~2,000-image shards with manifests. Resumable.
Gate 2: all shards complete, nearest-neighbour check passes.

Needs a GPU (L4/T4) and Tesseract. Paths assume the Drive layout in `DECISIONS.md`; adjust `D` for your account.

In [ ]:
# Phase 2 — setup: mount Drive, clone the public repo. Needs a GPU (L4/T4) + Tesseract.
import os
os.chdir("/content")

from google.colab import drive
drive.mount("/content/drive")

REPO = "/content/laya-vision"
if not os.path.exists(REPO):
    !git clone -q https://github.com/savita10/laya-vision.git {REPO}
else:
    !git -C {REPO} fetch -q origin && git -C {REPO} reset --hard origin/main
os.chdir(REPO)
!git log --oneline -1

!apt-get -qq install -y tesseract-ocr > /dev/null
!pip install -q "transformers==4.49.0" "laya==0.3.24" datasets pillow pytesseract

D = "/content/drive/MyDrive/laya-vision"
P1, P2, P3, STEP5 = f"{D}/phase1", f"{D}/phase2", f"{D}/phase3", f"{D}/step5"
for p in (P1, P2, P3): os.makedirs(p, exist_ok=True)

import torch, pytesseract
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0), "| tesseract", pytesseract.get_tesseract_version())

## Benchmark (64 val images)
Measures SigLIP and OCR throughput and projects hours for 70k. Result on an L4 with 12 CPUs: SigLIP 32 img/s,
Tesseract 7.3 img/s on 12 workers, EasyOCR 0.78 img/s — Tesseract confirmed. (`--ocr-engine all` also times EasyOCR;
install it only if you want that comparison.)

In [ ]:
!python scripts/bench_precompute.py --records {P1}/splits/val.jsonl --limit 64 --ocr-engine tesseract --out {P2}/bench_precompute.json

## Dry run into a scratch folder
Checks sharding, resume logging and the manifest before the real run. A `--limit` run counts as a different setup, so
it must use its own `--out-dir`.

In [ ]:
!python scripts/precompute.py --records {P1}/splits/val.jsonl --out-dir {P2}/scratch --limit 64
import json
m = json.load(open(f"{P2}/scratch/val/shard_0000/manifest.json")); m.pop("ids", None); m.pop("image_path", None)
print(json.dumps(m, indent=2)[:3000])

## Val, then the nearest-neighbour sanity check
val (5,000 images) takes ~17 min. The check asks whether same-class documents cluster in the frozen embedding space
before any training. Result: 1-NN label agreement 0.756 (12× chance), 5-NN 0.776 — above the 0.40 bar, so training is
worth attempting.

In [ ]:
!python scripts/precompute.py --records {P1}/splits/val.jsonl --out-dir {P2}/shards
!python scripts/nn_sanity_check.py --shards {P2}/shards/val --records {P1}/splits/val.jsonl --out {P2}/nn_sanity_val.json

## Test, then train
test ~15 min; train (60,000 images, 30 shards) ~2 h on an L4. Resumable: on a disconnect, re-run Cell A and this cell;
finished shards are skipped. The test split is processed under the recorded exception (image transforms only; labels
are dropped at parse time).

In [ ]:
for split in ["test", "train"]:
    print(f"\n===== {split} =====")
    !python scripts/precompute.py --records {P1}/splits/{split}.jsonl --out-dir {P2}/shards

## Gate 2 check
Expect 3 / 3 / 30 shards and ~7 GB.

In [ ]:
!ls {P2}/shards/val | grep -c shard
!ls {P2}/shards/test | grep -c shard
!ls {P2}/shards/train | grep -c shard
!du -sh {P2}/shards

Next: `phase3_colab_launcher.ipynb` (training).